# Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn transformers torch torchvision Pillow python-multipart omegaconf

# Create Config File

In [ ]:
yaml_config =  """
               model_path: "microsoft/table-transformer-detection"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

# Build Model

In [ ]:
import torch
import io
from PIL import Image
from omegaconf import OmegaConf
from transformers import AutoImageProcessor, AutoModelForObjectDetection


class TableDetection:
  def __init__(self, config_path):
    self.config = OmegaConf.load(config_path)
    self.processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-detection")
    self.model = AutoModelForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

  def __call__(self, imageBytes):
    # chuyển ảnh về dạng RGB
    image = Image.open(io.BytesIO(imageBytes)).convert("RGB")
    # Tiền xử lý
    input = self.processor(images = image, return_tensors = "pt")
    with torch.no_grad():
        output = self.model(**input)

    target_sizes = torch.tensor([image.size[::-1]])
    results = self.processor.post_process_object_detection(
                output, threshold=0.9, target_sizes=target_sizes
            )[0]

    # Trích xuất kết quả ra list dictionary
    detected_tables = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
      detected_tables.append({
        "label": self.model.config.id2label[label.item()],
        "confidence": round(score.item(), 3),
        "bounding_box": [round(i, 2) for i in box.tolist()]
      })

    return detected_tables

# Initialize Model

In [ ]:
detector = TableDetection("./config.yaml")

# Initialize API

In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


## TODO
@app.get('/')
async def root():
    return {"message": "Đồ án 1 - API Nhận diện bảng biểu (Table Detection)"}

@app.get('/health')
async def health():
    return {"status": "ok", "model": "microsoft/table-transformer-detection"}

@app.post('/predict')
async def predict(file: UploadFile = File(...)):
    try:
        # Đọc dữ liệu binary của ảnh
        imageBytes = await file.read()
        # Gọi model xử lý
        results = detector(imageBytes)
        return {
            "filename": file.filename,
            "detected_objects": results
        }
    except Exception as e:
        return {"error": str(e)}

## ENDTODO

import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("Server started on http://0.0.0.0:8000")

# Call Local API

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/predict"

with open("test4.jpg", "rb") as image_file:
    files = {"file": image_file}
    response = requests.post(API_URL, files=files)

print(response.json())